# ATiG 2026: LT-FH exercise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bvilhjal/ATIG_2026/blob/main/teaching_days/2026-09-24/LTFH/LTFH_exercise.ipynb)

In this exercise you simulate a disease register in which the true genetic liability of
every person is known, describe the data, score everyone with LT-FH++ using the Python
package [ltpred](https://github.com/bvilhjal/ltpred), and check the scores against the
truth.

Open the notebook in Colab, choose *File → Save a copy in Drive*, and run the cells in
order. Where a cell contains `...`, replace it with the expression described in the
comment next to it; a hint is given under each cell. Work in pairs. Parts 0, A and B are
for the class (about an hour); parts C to E are homework.

Background: under the liability-threshold model a person is diagnosed when their
liability, a genetic part `g` plus an environmental part, crosses a threshold. In
LT-FH++ ([Pedersen et al. 2022](https://doi.org/10.1016/j.ajhg.2022.01.009)) the
threshold depends on age and sex, and the score is the expected value of `g` given the
diagnoses and ages of a person and their relatives. See the slides from 22 September.

Python you will need: `x[mask]` keeps the entries of `x` where `mask` is True; `a & b` and
`~a` combine and negate True/False arrays; `.mean()` of a True/False array is the share of
True values; `[f(x) for x in xs]` applies `f` to each element of a list.

## Setup

In [ ]:
%pip install -q git+https://github.com/bvilhjal/ltpred@v0.7.1     # installs ltpred (about 30 s)

In [1]:
import numpy as np
from scipy.stats import norm, rankdata
import matplotlib.pyplot as plt

import ltpred
from ltpred import (simulate_pedigree, simulate_register_liabilities, estimate_liabilities,
                    kaplan_meier_cip)
from dataclasses import replace
from collections import Counter
print("ltpred", ltpred.__version__)


def corr(x, y):
    """Pearson correlation of two arrays."""
    return np.corrcoef(x, y)[0, 1]


def auc(score, case):
    """AUC: the chance that a random case scores above a random non-case.

    Computed from ranks (the Mann-Whitney statistic); `case` is a boolean array."""
    r = rankdata(score)
    n1, n0 = case.sum(), (~case).sum()
    return (r[case].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

ltpred 0.7.1


## Part 0: The data

The next cell simulates three generations of families. The disease affects 12% of men
and 8% of women by old age, h² = 0.5, and everyone is followed to age 70. Fathers are men
and mothers women; the others are given a random sex. `reg` holds the pedigree
(`reg.ids`, `reg.father`, `reg.mother`), whether each person was diagnosed by 70
(`reg.status`), the age at diagnosis (`reg.onset`), the age at the end of follow-up
(`reg.age`), the birth year (`reg.birth_time`) and the true genetic liability, stored as
`g`.

In [2]:
H2 = 0.5                                        # true liability-scale heritability
AGES = np.arange(0, 121.0)                      # ages 0, 1, ..., 120
CIP_M = 0.12 / (1 + np.exp((58 - AGES) / 8))    # men: lifetime 12%, half of it by age 58
CIP_F = 0.08 / (1 + np.exp((62 - AGES) / 8))    # women: lifetime 8%, half of it by age 62
K = 0.10                                        # lifetime prevalence, men and women together

ids, father, mother = simulate_pedigree(np.random.default_rng(1), n_founder_pairs=500, gens=2)
fathers, mothers = set(father), set(mother)
coin = np.random.default_rng(2).random(len(ids)) < 0.5  # a random sex for people who never became parents
male = np.array([p in fathers or (p not in mothers and c) for p, c in zip(ids, coin)])
sex = np.where(male, "M", "F")


def simulate(cip):
    """The register with one incidence curve for everyone (same seed = same liabilities)."""
    return simulate_register_liabilities(np.random.default_rng(1), ids, father, mother,
                                         h2=H2, cip_ages=AGES, cip_values=cip, eval_age=70)


as_men, as_women = simulate(CIP_M), simulate(CIP_F)
# each person's records follow their own sex's curve: sex-specific thresholds, as in LT-FH++
reg = replace(as_men, status=np.where(male, as_men.status, as_women.status),
              age=np.where(male, as_men.age, as_women.age),
              onset=np.where(male, as_men.onset, as_women.onset))
g = reg.genetic                                 # the truth, known only because we simulated it

print(f"{len(reg.ids)} people ({male.sum()} men), {reg.status.sum()} diagnosed by age 70")
print(f"diagnosed: men {reg.status[male].mean():.1%}, women {reg.status[~male].mean():.1%}")

5075 people (2482 men), 366 diagnosed by age 70
diagnosed: men 9.0%, women 5.5%


### Q1: When were people born, and how large are the families?

In [ ]:
row = {p: i for i, p in enumerate(reg.ids)}                      # id -> row number
years, n_born = np.unique(reg.birth_time, return_counts=True)    # people born in each year
couples = Counter((f, m) for f, m in zip(reg.father, reg.mother) if f in row and m in row)
sizes, n_couples = np.unique(list(couples.values()), return_counts=True)   # children per couple

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.bar(years, ..., width=6)          # how many people were born in each year: n_born
ax1.set_xlabel("birth year")
ax1.set_ylabel("people")
ax2.bar(sizes, ...)                   # how many couples have each number of children: n_couples
ax2.set_xlabel("children per couple")
ax2.set_ylabel("couples")
plt.show()

<details><summary>Hint</summary>

`n_born` and `n_couples`.

</details>

### Q2: At what ages are men and women diagnosed?

In [ ]:
bins = np.arange(0, 75, 5)                                      # 5-year age bins
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(reg.onset[male & reg.status], bins=bins, alpha=0.6, label="men")
ax.hist(..., bins=bins, alpha=0.6, label="women")               # the women's ages at diagnosis
ax.set_xlabel("age at diagnosis")
ax.set_ylabel("cases")
ax.legend()
plt.show()
print(f"cases: {(male & reg.status).sum()} men, {(~male & reg.status).sum()} women; median age at "
      f"diagnosis {np.median(reg.onset[male & reg.status]):.0f} and {np.median(reg.onset[~male & reg.status]):.0f}")

<details><summary>Hint</summary>

`reg.onset[~male & reg.status]`

</details>

### Q3: Plot the Kaplan–Meier survival and the cumulative incidence by sex.

The survival curve S(t) is the share still undiagnosed at age t; the cumulative incidence
is 1 − S(t). `kaplan_meier_cip(entry, exit, event)` estimates the cumulative incidence. Everyone enters at
birth and leaves at diagnosis or at 70. Compare the observed curves with the true ones
(dashed).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for label, keep, cip, colour in (("men", male, CIP_M, "C0"), ("women", ~male, CIP_F, "C1")):
    # Kaplan-Meier: everyone enters at birth and leaves at diagnosis or at 70
    km = kaplan_meier_cip(np.zeros(keep.sum()), ..., ...)    # exit ages and events: reg.age[keep], reg.status[keep]
    ages, inc = np.r_[0, km.ages], np.r_[0, km.values]      # start both curves at age 0
    ax1.step(ages, 100 * (1 - inc), where="post", color=colour, label=label)   # survival S(t)
    ax2.step(ages, 100 * inc, where="post", color=colour, label=f"{label}: observed")
    ax2.plot(AGES[:71], 100 * cip[:71], ls="--", color=colour, label=f"{label}: true curve")
ax1.set_title("Kaplan-Meier survival S(t)")
ax1.set_ylabel("% still undiagnosed")
ax2.set_title("cumulative incidence = 1 - S(t)")
ax2.set_ylabel("% diagnosed by this age")
for ax in (ax1, ax2):
    ax.set_xlabel("age")
    ax.legend()
plt.show()

<details><summary>Hint</summary>

`reg.age[keep]` and `reg.status[keep]`.

</details>

### Q4: Does a diagnosed parent change the cumulative incidence?

In [ ]:
row = {p: i for i, p in enumerate(reg.ids)}                  # id -> row number
has_parents = np.array([f in row for f in reg.father])       # parents recorded in the register
parent_dx = np.array([any(reg.status[row[p]] for p in (f, m) if p in row)
                      for f, m in zip(reg.father, reg.mother)])   # a parent diagnosed by 70

fig, ax = plt.subplots(figsize=(6.5, 4))
for label, keep in (("a diagnosed parent", parent_dx),
                    ("no diagnosed parent", ...)):          # parents recorded, but neither diagnosed
    km = kaplan_meier_cip(np.zeros(keep.sum()), reg.age[keep], reg.status[keep])
    ax.step(km.ages, 100 * km.values, where="post", label=f"{label} (n = {keep.sum()})")
ax.set_xlabel("age")
ax.set_ylabel("% diagnosed by this age")
ax.legend()
plt.show()

<details><summary>Hint</summary>

`has_parents & ~parent_dx`

</details>

### Q5: How different is the true `g` of cases and non-cases?

This can only be plotted in a simulation.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(g[~reg.status], bins=50, density=True, alpha=0.5, label="not diagnosed by 70")
ax.hist(..., bins=50, density=True, alpha=0.5, label="diagnosed by 70")    # g of the cases
ax.set_xlabel("true genetic liability g")
ax.legend()
plt.show()
print(f"mean g: cases {g[reg.status].mean():.2f}, non-cases {g[~reg.status].mean():.2f}; "
      f"of the people with g > 1, {reg.status[g > 1].mean():.0%} were diagnosed")

<details><summary>Hint</summary>

`g[reg.status]`

</details>

### Scoring

`estimate_liabilities` returns each person's score (`.est`, the expected `g` given the
records) and its variance (`.var`). `strata=sex` gives each sex its own incidence curve.
The helpers below rescore everyone under another h² (`score`) and score people as of their
40th birthday, using only records made before it (`score_at_40`).

In [3]:
curves = {"M": (AGES, CIP_M, 0.12), "F": (AGES, CIP_F, 0.08)}   # sex -> (ages, curve, lifetime K)

scores = estimate_liabilities(
    reg.ids, reg.father, reg.mother,          # the pedigree: who is whose parent
    probands=reg.ids,                         # whom to score: everyone
    status=reg.status, age=reg.age,           # diagnosed by 70? age at diagnosis or at 70
    strata=sex, cip_by_stratum=curves,        # each sex has its own incidence curve, so its own thresholds
    h2=H2,                                    # heritability: you supply it, the scorer never estimates it
    use="gwas")                               # also use each person's own diagnosis

est, var = scores.est, scores.var
print(f"corr(score, g) = {corr(est, g):.3f}")
print(f"var(score) = {est.var():.3f}  +  mean posterior variance = {var.mean():.3f}"
      f"  =  {est.var() + var.mean():.3f}")

corr(score, g) = 0.529
var(score) = 0.128  +  mean posterior variance = 0.359  =  0.488


In [4]:
free40 = reg.onset > 40                     # still undiagnosed on their 40th birthday


def score(h2):
    """Everyone's score from all records up to 70 (use="gwas"): (mean, variance)."""
    s = estimate_liabilities(reg.ids, reg.father, reg.mother, probands=reg.ids,
                             status=reg.status, age=reg.age, strata=sex, cip_by_stratum=curves,
                             h2=h2, use="gwas")
    return s.est, s.var


def score_at_40(h2):
    """Scores as known on each person's 40th birthday (use="prediction"), for the
    people still undiagnosed then: their own status and every later record are hidden."""
    s = estimate_liabilities(reg.ids, reg.father, reg.mother,
                             probands=[p for p, keep in zip(reg.ids, free40) if keep],
                             status=reg.status, age=reg.age, strata=sex, cip_by_stratum=curves,
                             h2=h2, use="prediction",
                             birth_time=reg.birth_time,                  # everyone's birth date
                             index_time=(reg.birth_time + 40)[free40])   # each proband's 40th birthday
    return s.est, s.var


est40, var40 = score_at_40(H2)
print(f"{free40.sum()} people undiagnosed at 40; corr(score at 40, g) = {corr(est40, g[free40]):.3f}")

5040 people undiagnosed at 40; corr(score at 40, g) = 0.280


### Q6: Plot the score against the true `g`.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(..., ..., s=3, alpha=0.4)           # x: the score est, y: the truth g
ax.set_xlabel("score (posterior mean)")
ax.set_ylabel("true genetic liability g")
ax.set_title(f"corr = {corr(est, g):.2f}")
plt.show()

<details><summary>Hint</summary>

`ax.scatter(est, g, ...)`

</details>

## Part A: A wrong heritability

### Q7: Score the register assuming h² = 0.2, 0.5 and 0.8.

Before running the cell, guess whether the scores spread more or less when h² is set too
high, and whether their ranking changes. The score is calibrated if the slope of `g` on
the score is 1.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True, sharey=True)
for ax, h2 in zip(axes, (0.2, 0.5, 0.8)):
    e, v = score(h2)                           # rescore everyone under this h2
    b = ...                                    # slope of g regressed on e: np.polyfit(e, g, 1)[0]
    ax.scatter(e, g, s=2, alpha=0.3)
    ax.axline((0, 0), slope=1, ls="--", color="grey")   # slope 1: calibrated
    ax.axline((0, 0), slope=b, color="C1")              # the fitted slope
    ax.set_title(f"assumed h² = {h2}\ncorr {corr(e, g):.3f}, slope {b:.2f}")
    ax.set_xlabel("score")
axes[0].set_ylabel("true g")
plt.show()

<details><summary>Hint</summary>

`np.polyfit(e, g, 1)[0]`

</details>

## Part B: Predicting later diagnosis

![Liability thresholds at ages 40 and 70.](https://bvilhjal.github.io/ATIG_2026/teaching_days/2026-09-24/LTFH/figures/thresholds.png)

We now use the score at 40 (`est40`) for the people undiagnosed at 40, and ask who is
diagnosed between 40 and 70.

In [5]:
y = reg.status[free40]      # diagnosed between 40 and 70, one entry per person in est40
print(f"{y.sum()} of {len(y)} people undiagnosed at 40 were diagnosed by 70")

331 of 5040 people undiagnosed at 40 were diagnosed by 70


### Q8: Compare the scores of later cases and non-cases.

Plot the two histograms and compute the AUC for the score at 40, for the true `g` and for
the score that used each person's own diagnosis (`use="gwas"`).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(est40[~y], bins=40, density=True, alpha=0.5, label="not diagnosed")
ax.hist(..., bins=40, density=True, alpha=0.5, label="diagnosed 40 to 70")   # the cases' scores
ax.set_xlabel("score at 40")
ax.legend()
plt.show()

print(f"AUC score at 40        {auc(est40, y):.3f}")
print(f"AUC true g             {auc(..., y):.3f}")      # g for the same people
print(f"AUC use='gwas' score   {auc(est[free40], y):.3f}")

<details><summary>Hint</summary>

`est40[y]` for the histogram and `g[free40]` for the AUC.

</details>

### Q9: Convert the score into a risk and compare it with what happened.

The cell computes each person's risk of diagnosis between 40 and 70 from the score, its
variance and the thresholds for their sex at 40 and 70.

In [ ]:
cip40 = np.where(male, np.interp(40, AGES, CIP_M), np.interp(40, AGES, CIP_F))[free40]
cip70 = np.where(male, np.interp(70, AGES, CIP_M), np.interp(70, AGES, CIP_F))[free40]
T40, T70 = norm.isf(cip40), norm.isf(cip70)            # each person's LT-FH++ thresholds at 40 and 70
sd = np.sqrt(var40 + 1 - H2)                           # spread of full liability given the relatives
below40 = norm.cdf((T40 - est40) / sd)                 # P(undiagnosed at 40)
below70 = norm.cdf((T70 - est40) / sd)                 # P(undiagnosed at 70)
risk = (below40 - below70) / below40                   # P(diagnosed 40-70 | undiagnosed at 40)

fifths = np.array_split(np.argsort(est40, kind="stable"), 5)   # lowest to highest score
observed = [y[idx].mean() for idx in fifths]                   # share diagnosed in each fifth
predicted = [...]                                              # mean risk in each fifth

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(np.arange(5) - 0.2, observed, width=0.4, label="observed")
ax.bar(np.arange(5) + 0.2, predicted, width=0.4, label="predicted risk")
ax.set_xticks(range(5), ["lowest", "2nd", "middle", "4th", "highest"])
ax.set_xlabel("fifth of the score at 40")
ax.set_ylabel("share diagnosed 40 to 70")
ax.legend()
plt.show()
print(f"mean predicted risk {risk.mean():.3f}   observed {y.mean():.3f}")

<details><summary>Hint</summary>

Copy the `observed` line with `risk[idx]` in place of `y[idx]`.

</details>

## Part C: The score against a simple family history (homework)

The next cell finds each person's parents and siblings and marks who has a diagnosed one,
by age 70 (`fh`) and by their own 40th birthday (`fh40`).

In [6]:
row = {p: i for i, p in enumerate(reg.ids)}          # id -> row number
children = {}                                        # (father, mother) -> rows of their children
for i, (f, m) in enumerate(zip(reg.father, reg.mother)):
    if f in row and m in row:
        children.setdefault((f, m), []).append(i)


def first_degree(i):
    """Rows of person i's parents and full siblings."""
    f, m = reg.father[i], reg.mother[i]
    parents = [row[p] for p in (f, m) if p in row]
    siblings = [j for j in children.get((f, m), []) if j != i]
    return np.array(parents + siblings, dtype=int)


fdr = [first_degree(i) for i in range(len(reg.ids))]
diag_time = reg.birth_time + reg.onset               # calendar time of each diagnosis
birth40 = reg.birth_time + 40                        # calendar time of each 40th birthday

# yes/no family history: any affected parent or sibling, by 70 and by one's own 40th birthday
fh = np.array([reg.status[r].any() for r in fdr])
fh40 = np.array([(reg.status[r] & (diag_time[r] <= birth40[i])).any() for i, r in enumerate(fdr)])
print(f"family-history positive: {fh.sum()} by age 70, {fh40.sum()} at their 40th birthday")

family-history positive: 971 by age 70, 627 at their 40th birthday


### Q10: How much of `g` do own status, family history and the score explain?

In [ ]:
names = ["own status", "FH indicator", "score (use='gwas')"]
r2 = [... for x in (reg.status, fh, est)]      # R²: the squared correlation of each with g

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(names, r2)
ax.set_ylabel("R² with the true g")
plt.show()
print(f"prediction at 40: AUC FH indicator {auc(fh40[free40], y):.3f}, AUC score {auc(est40, y):.3f}")

<details><summary>Hint</summary>

`corr(x, g) ** 2`

</details>

## Part D: Heritability from parents and children (homework)

h² is roughly twice the tetrachoric correlation between the diagnoses of parents and
children. The next cell lists all parent–child pairs.

In [7]:
from ltpred import tetrachoric

# every parent-child pair in the register, as row numbers
pairs = [(row[p], i) for i, (f, m) in enumerate(zip(reg.father, reg.mother))
         for p in (f, m) if p in row]
par, kid = np.array(pairs).T


def table(a, b):
    """2x2 table of two True/False arrays: both, first only, second only, neither."""
    return np.array([(a & b).sum(), (a & ~b).sum(), (~a & b).sum(), (~a & ~b).sum()])

### Q11: Estimate h² from the parent–child pairs.

In [ ]:
p_status, k_status = reg.status[par], reg.status[kid]
t = ...                      # tetrachoric(first status array, second status array)
print(f"{len(par)} pairs, table {table(p_status, k_status)}")
print(f"h2 = 2 rho = {2 * t.rho:.2f} +/- {2 * t.se:.2f}   (truth {H2})")

<details><summary>Hint</summary>

`tetrachoric(p_status, k_status)`

</details>

### Q12: What would make this estimate wrong in a real register?

Think about follow-up, sex, shared environment and how people were sampled.

## Part E: Observed and liability scale (homework)

![A liability split into a 0/1 outcome.](https://bvilhjal.github.io/ATIG_2026/teaching_days/2026-09-24/LTFH/figures/observed_scale.png)

GWAS methods report h² on the observed 0/1 scale, which depends on the proportion of
cases in the sample ([Lee et al. 2011](https://doi.org/10.1016/j.ajhg.2011.02.002)).

### Q13: Plot the observed-scale h² against the case fraction for a true h² of 0.5.

In [ ]:
from ltpred import liability_to_observed_h2

Ps = np.linspace(0.02, 0.6, 50)                # case fraction in the sample
h2_obs = [...]                                 # observed-scale h² for each P: liability_to_observed_h2(H2, K, P)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(Ps, h2_obs)
ax.axhline(H2, ls="--", color="grey", label="liability scale: 0.5")
ax.axvline(K, ls=":", color="C1", label="population sample: P = K")
ax.set_xlabel("case fraction in the sample, P")
ax.set_ylabel("observed-scale h²")
ax.legend()
plt.show()
print(f"population sample {float(liability_to_observed_h2(H2, K, None)):.3f}, "
      f"1:1 study {float(liability_to_observed_h2(H2, K, 0.5)):.3f}")

<details><summary>Hint</summary>

`[float(liability_to_observed_h2(H2, K, P)) for P in Ps]`

</details>

### Q14: Summarise

In about 100 words, say what the score estimates, what you found, and what it cannot
tell you.